In [1]:
import os
import pandas as pd

GOLD_PATH = './Medallion Architecture/gold/'
os.makedirs(GOLD_PATH, exist_ok=True)


In [2]:
import importlib

parquet_engine = None
for candidate in ('pyarrow', 'fastparquet'):
    if importlib.util.find_spec(candidate):
        parquet_engine = candidate
        break

if parquet_engine is None:
    raise ImportError(
        "Parquet support is missing. Install `pyarrow` or `fastparquet` in this notebook environment, then restart the kernel."
    )

print(f'Parquet engine available: {parquet_engine}')


Parquet engine available: pyarrow


In [3]:
df = pd.read_parquet('./Medallion Architecture/silver/silver_transactions.parquet')

In [4]:
df.head()

,Player Name,Team,World,Vehicle Type,Companion,Kart Racing Rank,Platforming Rank,Boss Battle Rank,Power-Ups Used,Kart Role,...,Lives Lost,Participation in Battle Mode,Mushroom Cup Participation,Power-Ups Owned,Coins Spent in Toad Town,Levels Completed,Times Hit by Enemies,Primary Game,fileName,loadDatetimeStamp
0,Yoshi,Toad Brigade,Yoshi's Island,Comet Bike,Polterpup,A,A,A,12,Drifter,...,0,True,False,1-Up Mushroom,64,26,4,Mario Tennis Aces,mario_data_20250101.csv,2026-03-18 11:24:40.640227
1,Peach,Green Caps,Donut Plains,Circuit Special,Koopa Troopa,C,A,B,16,Drifter,...,4,False,False,"Red Shell, Super Star",335,40,5,Mario Tennis Aces,mario_data_20250101.csv,2026-03-18 11:24:40.640227
2,Waluigi,No Team,Yoshi's Island,Biddybuggy,Goomba,D,A,C,26,Blocker,...,1,False,True,Green Shell,182,57,6,Mario Kart 8 Deluxe,mario_data_20250101.csv,2026-03-18 11:24:40.640227
3,Yoshi,Toad Brigade,Star World,Pipe Frame,Goomba,C,D,A,23,Drifter,...,5,False,True,1-Up Mushroom,333,84,6,Super Mario Bros.,mario_data_20250101.csv,2026-03-18 11:24:40.640227
4,Bowser Jr.,Koopa Clan,Mushroom Kingdom,Pipe Frame,Toad,C,C,B,10,Blocker,...,2,True,False,"Red Shell, Banana Peel, Fire Flower",461,55,7,Super Mario World,mario_data_20250101.csv,2026-03-18 11:24:40.640227


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48000 entries, 0 to 47999
Data columns (total 21 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   Player Name                   48000 non-null  string        
 1   Team                          48000 non-null  string        
 2   World                         48000 non-null  string        
 3   Vehicle Type                  48000 non-null  string        
 4   Companion                     48000 non-null  string        
 5   Kart Racing Rank              48000 non-null  string        
 6   Platforming Rank              48000 non-null  string        
 7   Boss Battle Rank              48000 non-null  string        
 8   Power-Ups Used                48000 non-null  int32         
 9   Kart Role                     48000 non-null  string        
 10  Team Points                   48000 non-null  int32         
 11  Lives Lost                  

In [6]:
rank_map = {'S': 5, 'A': 4, 'B': 3, 'C': 2, 'D': 1}
df['Kart_Racing_Rank_Score'] = df['Kart Racing Rank'].map(rank_map)

team_report = (
    df.groupby('Team')
      .agg({
          'Team Points': 'sum',
          'Kart_Racing_Rank_Score': 'mean',
          'Coins Spent in Toad Town': 'sum'
      })
      .reset_index()
)

team_report.columns = ['Team', 'Total Team Points', 'Average Rank Score', 'Total Coins Spent']
team_report = team_report.sort_values(by='Total Team Points', ascending=False)

print('Step 1 complete — Team Performance Summary:\n', team_report.head(5))

team_report.to_parquet('./Medallion Architecture/gold/gold_team_summary.parquet', index=False)

Step 1 complete — Team Performance Summary:
                Team  Total Team Points  Average Rank Score  Total Coins Spent
7      Toad Brigade             290470            3.040237            1395065
6  Sarasaland Stars             289719            3.030422            1426623
3  Mushroom Royalty             284529            2.999568            1438670
1        Green Caps             283362            3.012155            1411585
2        Koopa Clan             282639            2.987751            1375545


In [7]:
player_points = (
    df.groupby('Player Name')['Team Points']
      .sum()
      .reset_index()
      .sort_values(by='Team Points', ascending=False)
)

print('Step 2 complete — Total Team Points per Player:\n', player_points.head(5))

player_points.to_parquet('./Medallion Architecture/gold/gold_player_performance.parquet', index=False)

Step 2 complete — Total Team Points per Player:
        Player Name  Team Points
9   Unknown Player       454832
3            Luigi       172454
1       Bowser Jr.       170935
0           Bowser       165806
10         Waluigi       165660


In [8]:
powerups_report = (
    df.groupby('Player Name')['Power-Ups Used']
      .agg(['count', 'mean'])
      .reset_index()
      .rename(columns={'count': 'Power-Up Count', 'mean': 'Average Power-Ups Used'})
      .sort_values(by='Average Power-Ups Used', ascending=False)
)

print('\nStep 3 complete — Power-Up Usage per Player:\n', powerups_report.head(10))

powerups_report.to_parquet('./Medallion Architecture/gold/gold_powerup_player.parquet', index=False)


Step 3 complete — Power-Up Usage per Player:
        Player Name  Power-Up Count  Average Power-Ups Used
6         Rosalina            3234               17.754484
7             Toad            3210               17.631776
9   Unknown Player            9040               17.603540
1       Bowser Jr.            3241               17.525455
12           Yoshi            3304               17.486077
2            Daisy            3294               17.472981
0           Bowser            3268               17.407895
8         Toadette            3297               17.397028
3            Luigi            3265               17.390199
10         Waluigi            3325               17.372932


In [9]:
world_difficulty = (
    df.groupby('World')['Lives Lost']
      .mean()
      .reset_index()
      .rename(columns={'Lives Lost': 'Average Lives Lost'})
      .sort_values(by='Average Lives Lost', ascending=False)
)

print('Step 4 complete — Average Lives Lost per World:\n', world_difficulty.head(10))

world_difficulty.to_parquet('./Medallion Architecture/gold/gold_world_difficulty.parquet', index=False)

Step 4 complete — Average Lives Lost per World:
               World  Average Lives Lost
0      Donut Plains            2.599926
2  Mushroom Kingdom            2.592742
4         Toad Town            2.581164
5    Yoshi's Island            2.572454
3        Star World            2.566345
1     Koopa Kingdom            2.546904


In [10]:
top_player_per_team = (
    df.loc[df.groupby('Team')['Team Points'].idxmax(), ['Team', 'Player Name', 'Team Points']]
    .sort_values(by='Team Points', ascending=False)
    .reset_index(drop=True)
)

print('Step 5 complete — Top Player per Team:\n', top_player_per_team.head(10))

top_player_per_team.to_parquet('./Medallion Architecture/gold/gold_top_player_per_team.parquet', index=False)

Step 5 complete — Top Player per Team:
                Team Player Name  Team Points
0        Koopa Clan       Yoshi          201
1          Red Caps      Bowser          201
2      Dino Buddies     Waluigi          200
3        Green Caps       Mario          200
4  Mushroom Royalty      Bowser          200
5           No Team     Waluigi          200
6  Sarasaland Stars     Waluigi          200
7      Toad Brigade       Peach          200
8        Tricksters     Waluigi          200


In [11]:
vehicle_counts = (
    df.groupby('Vehicle Type')
    .size()
    .reset_index(name='Count')
    .sort_values(by='Count', ascending=False)
)

print('Step 6 complete — Vehicle Popularity Report:\n', vehicle_counts.head(10))

vehicle_counts.to_parquet('./Medallion Architecture/gold/gold_vehicle_counts.parquet', index=False)

Step 6 complete — Vehicle Popularity Report:
       Vehicle Type  Count
5          Scooter   6916
6    Standard Kart   6914
4       Pipe Frame   6868
0       Biddybuggy   6863
3           Mach 8   6839
2       Comet Bike   6828
1  Circuit Special   6772


In [12]:
risk_assessment = (
    df.groupby('Player Name')['Lives Lost']
    .sum()
    .reset_index()
    .sort_values(by='Lives Lost', ascending=False)
)

print('Step 7 complete — Risk Assessment (Players with Most Lives Lost):\n', risk_assessment.head(10))

risk_assessment.to_parquet('./Medallion Architecture/gold/gold_risk_assessment.parquet', index=False)

Step 7 complete — Risk Assessment (Players with Most Lives Lost):
        Player Name  Lives Lost
9   Unknown Player       23323
12           Yoshi        8610
10         Waluigi        8591
8         Toadette        8574
2            Daisy        8425
1       Bowser Jr.        8405
5            Peach        8365
3            Luigi        8318
6         Rosalina        8315
0           Bowser        8298


In [13]:
world_completion = (
    df.groupby('World')['Levels Completed']
    .agg(['mean', 'sum'])
    .reset_index()
    .rename(columns={'mean': 'Average Levels Completed', 'sum': 'Total Levels Completed'})
    .sort_values(by='Average Levels Completed', ascending=False)
)

print('Step 8 complete — World Completion Summary:\n', world_completion.head(10))

world_completion.to_parquet('./Medallion Architecture/gold/gold_world_completion.parquet', index=False)

Step 8 complete — World Completion Summary:
               World  Average Levels Completed  Total Levels Completed
0      Donut Plains                 62.734232                  507269
1     Koopa Kingdom                 62.687679                  503194
5    Yoshi's Island                 62.572580                  497014
4         Toad Town                 62.501433                  501699
3        Star World                 62.248966                  496809
2  Mushroom Kingdom                 61.904864                  491277


In [14]:
hits_team = (
    df.groupby('Team')['Times Hit by Enemies']
    .agg(['mean', 'sum'])
    .reset_index()
    .rename(columns={'mean': 'Average Hits by Enemies', 'sum': 'Total Hits by Enemies'})
    .sort_values(by='Total Hits by Enemies', ascending=False)
)

print('Step 9 complete — Team Hit Summary:\n', hits_team.head(10))

hits_team.to_parquet('./Medallion Architecture/gold/gold_hits_team.parquet', index=False)

Step 9 complete — Team Hit Summary:
                Team  Average Hits by Enemies  Total Hits by Enemies
6  Sarasaland Stars                 5.088503                  29035
3  Mushroom Royalty                 5.026261                  28710
7      Toad Brigade                 5.010887                  28537
2        Koopa Clan                 5.093475                  28335
1        Green Caps                 5.022222                  28250
5          Red Caps                 5.013312                  28245
8        Tricksters                 4.990352                  27931
0      Dino Buddies                 5.034858                  27732
4           No Team                 5.111486                  15130


In [15]:
spending_analysis = (
    df.groupby('Team')['Coins Spent in Toad Town']
    .sum()
    .reset_index()
    .rename(columns={'Coins Spent in Toad Town': 'Total Coins Spent'})
    .sort_values(by='Total Coins Spent', ascending=False)
)

print('Step 10 complete — Spending Analysis (Coins Spent by Each Team):\n', spending_analysis.head(10))

spending_analysis.to_parquet('./Medallion Architecture/gold/gold_spending_analysis.parquet', index=False)

Step 10 complete — Spending Analysis (Coins Spent by Each Team):
                Team  Total Coins Spent
3  Mushroom Royalty            1438670
6  Sarasaland Stars            1426623
1        Green Caps            1411585
5          Red Caps            1409952
7      Toad Brigade            1395065
8        Tricksters            1394614
2        Koopa Clan            1375545
0      Dino Buddies            1363819
4           No Team             741924
